# Satellite Building Change Detection (LEVIR-CD)

CS57600 Machine Learning, Fall 2026 · Raviteja Gundu, Joel Adu-Kwarteng

Predict pixel-level building-change masks from pairs of satellite images, comparing an image-difference baseline, an early-fusion U-Net (U-Net-EF) and a Siamese U-Net.

**How to run**
1. Upload this notebook to Colab and choose **Runtime → Change runtime type → T4 GPU**.
2. Run all cells. Allow Drive access when prompted.

Data, checkpoints and results are stored on Google Drive under `CONFIG["project_root"]`, so they survive between Colab sessions. Each task adds one section to this notebook.

## Task 1 · Setup

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib
import numpy as np
import PIL
import torch

# All settings live here. Later tasks add their own entries (data, baseline, models).
CONFIG = {
    "project_root": "/content/drive/MyDrive/levir-change-detection",  # persistent storage
    "local_root": "/content/local",  # fast VM disk, wiped when the session ends
    "seed": 42,
    "deterministic": False,  # True makes cuDNN deterministic (slower)
}

In [ ]:
def set_seed(seed: int, deterministic: bool = False) -> None:
    """Seed Python, NumPy and torch (CPU and all GPUs)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic


def seed_worker(worker_id: int) -> None:
    """DataLoader worker_init_fn: derive NumPy/Python seeds from torch's per-worker seed."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


@dataclass(frozen=True)
class Paths:
    """Standard locations under the Drive project root, plus a fast local scratch folder."""

    root: Path
    local: Path

    @property
    def raw(self) -> Path:  # LEVIR-CD as downloaded: train/val/test/{A,B,label}
        return self.root / "data" / "raw"

    @property
    def tiles_zip(self) -> Path:  # 256x256 tiles (Task 3)
        return self.root / "data" / "tiles.zip"

    @property
    def stats_file(self) -> Path:  # training-set channel mean/std (Task 4)
        return self.root / "data" / "stats.json"

    @property
    def runs(self) -> Path:  # checkpoints and logs, one folder per run
        return self.root / "runs"

    @property
    def results(self) -> Path:  # metrics JSON and CSVs
        return self.root / "results"

    @property
    def figures(self) -> Path:  # figures for the report
        return self.results / "figures"

    @property
    def local_tiles(self) -> Path:  # tiles unzipped to local disk each session
        return self.local / "tiles"

    def run_dir(self, name: str) -> Path:
        return self.runs / name

    def make_dirs(self) -> "Paths":
        for p in (self.raw, self.runs, self.results, self.figures, self.local):
            p.mkdir(parents=True, exist_ok=True)
        return self

In [ ]:
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive is not mounted. Run the drive.mount cell above first.")

set_seed(CONFIG["seed"], CONFIG["deterministic"])
DEVICE = get_device()
PATHS = Paths(Path(CONFIG["project_root"]), Path(CONFIG["local_root"])).make_dirs()

print(f"torch {torch.__version__} | numpy {np.__version__} | pillow {PIL.__version__} | matplotlib {matplotlib.__version__}")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Device:", DEVICE)
print()
for name in ("root", "raw", "tiles_zip", "stats_file", "runs", "results", "figures", "local_tiles"):
    print(f"{name:>11}: {getattr(PATHS, name)}")

## Task 2 · Metrics

All metrics are for the **change** class (label 1) and come from confusion counts:

| Metric | Formula |
|---|---|
| Precision | TP / (TP + FP) |
| Recall | TP / (TP + FN) |
| F1 (= Dice) | 2·TP / (2·TP + FP + FN) |
| IoU | TP / (TP + FP + FN) |
| Accuracy | (TP + TN) / all pixels |

Dataset-level results add up the counts over every test pixel first and compute the metrics once (not an average of per-image scores).

Zero denominators: if a mask has no change and the prediction has none either (TP = FP = FN = 0), precision, recall, F1 and IoU are 1.0. Any other zero denominator gives 0.0. This mostly matters for per-image scores on no-change tiles.

Inputs must already be binary (0/1 or bool). Threshold probabilities first. Tensors are counted on their own device, so validation on the GPU never copies masks to the CPU.

In [ ]:
METRIC_NAMES = ("precision", "recall", "f1", "iou", "accuracy")


def _squeeze_channel(x):
    """(N, 1, H, W) -> (N, H, W), so model outputs and masks line up."""
    return x[:, 0] if x.ndim == 4 and x.shape[1] == 1 else x


def _as_binary(x, name):
    """Return a bool tensor/array. Raises if values other than 0/1 are present."""
    if torch.is_tensor(x):
        if x.dtype != torch.bool:
            if not ((x == 0) | (x == 1)).all():
                raise ValueError(f"{name} must be binary (0/1). Threshold probabilities first.")
            x = x != 0
        return x
    x = np.asarray(x)
    if x.dtype != bool:
        if not np.isin(x, (0, 1)).all():
            raise ValueError(f"{name} must be binary (0/1). Threshold probabilities first.")
        x = x != 0
    return x


def confusion_counts(pred, target):
    """Return (tp, fp, fn, tn) as Python ints for binary masks of the same shape.

    Accepts numpy arrays or torch tensors. If either is a tensor, counting happens on
    that tensor's device.
    """
    if torch.is_tensor(pred) or torch.is_tensor(target):
        device = pred.device if torch.is_tensor(pred) else target.device
        pred = torch.as_tensor(pred, device=device)
        target = torch.as_tensor(target, device=device)
    pred, target = _squeeze_channel(pred), _squeeze_channel(target)
    if tuple(pred.shape) != tuple(target.shape):
        raise ValueError(f"Shape mismatch: pred {tuple(pred.shape)} vs target {tuple(target.shape)}")
    p, t = _as_binary(pred, "pred"), _as_binary(target, "target")
    tp = int((p & t).sum())
    fp = int((p & ~t).sum())
    fn = int((~p & t).sum())
    tn = int(t.numel() if torch.is_tensor(t) else t.size) - tp - fp - fn
    return tp, fp, fn, tn


def metrics_from_counts(tp, fp, fn, tn):
    """Precision, recall, F1, IoU and accuracy from confusion counts, plus the counts."""
    empty = tp + fp + fn == 0  # nothing to find and nothing predicted

    def ratio(num, den):
        if den == 0:
            return 1.0 if empty else 0.0
        return num / den

    total = tp + fp + fn + tn
    return {
        "precision": ratio(tp, tp + fp),
        "recall": ratio(tp, tp + fn),
        "f1": ratio(2 * tp, 2 * tp + fp + fn),
        "iou": ratio(tp, tp + fp + fn),
        "accuracy": (tp + tn) / total if total else 1.0,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }


class ConfusionAccumulator:
    """Running TP/FP/FN/TN over many batches. Call compute() for dataset-level metrics."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.tp = self.fp = self.fn = self.tn = 0

    def update(self, pred, target):
        tp, fp, fn, tn = confusion_counts(pred, target)
        self.tp += tp
        self.fp += fp
        self.fn += fn
        self.tn += tn
        return self

    def compute(self):
        return metrics_from_counts(self.tp, self.fp, self.fn, self.tn)

    def __repr__(self):
        return f"ConfusionAccumulator(tp={self.tp}, fp={self.fp}, fn={self.fn}, tn={self.tn})"


def per_image_metrics(pred, target, names=None):
    """Metrics for each image in a batch shaped (N, H, W) or (N, 1, H, W).

    Returns a list of dicts. If names is given, each dict also gets a "name" key.
    """
    pred, target = _squeeze_channel(pred), _squeeze_channel(target)
    if len(pred) != len(target):
        raise ValueError(f"Batch size mismatch: {len(pred)} vs {len(target)}")
    rows = []
    for i in range(len(pred)):
        row = metrics_from_counts(*confusion_counts(pred[i], target[i]))
        if names is not None:
            row = {"name": names[i], **row}
        rows.append(row)
    return rows


def format_metrics(m):
    return "  ".join(f"{k}={m[k]:.4f}" for k in METRIC_NAMES)

### Demo: hand-computed toy example

A 4×4 target with 4 changed pixels and a prediction with 4 changed pixels, 3 of them correct:
TP = 3, FP = 1, FN = 1, TN = 11, so precision = recall = F1 = 0.75, IoU = 3/5 = 0.6 and accuracy = 14/16 = 0.875.

The cell checks these values and a few other cases, and raises an error if anything is off.

In [ ]:
target = np.array([[0, 0, 0, 0],
                   [0, 1, 1, 0],
                   [0, 1, 1, 0],
                   [0, 0, 0, 0]])
pred = np.array([[0, 0, 0, 0],
                 [0, 1, 1, 1],
                 [0, 1, 0, 0],
                 [0, 0, 0, 0]])


def check(label, got, expected):
    ok = all(np.isclose(got[k], v) for k, v in expected.items())
    print(f"{'OK  ' if ok else 'FAIL'} {label:<38} {format_metrics(got)}")
    if not ok:
        raise AssertionError(f"{label}: expected {expected}, got {got}")


# 1. Hand-computed example
check("toy 4x4 (numpy)", metrics_from_counts(*confusion_counts(pred, target)),
      {"tp": 3, "fp": 1, "fn": 1, "tn": 11,
       "precision": 0.75, "recall": 0.75, "f1": 0.75, "iou": 0.6, "accuracy": 0.875})

# 2. Same example as torch tensors on the GPU, shaped like model output (N, 1, H, W)
t_pred = torch.tensor(pred, device=DEVICE)[None, None]
t_target = torch.tensor(target, device=DEVICE)[None]
check(f"toy 4x4 (torch on {DEVICE.type})", metrics_from_counts(*confusion_counts(t_pred, t_target)),
      {"precision": 0.75, "recall": 0.75, "f1": 0.75, "iou": 0.6})

# 3. Edge cases
zeros, ones = np.zeros((4, 4), int), np.ones((4, 4), int)
check("no change, nothing predicted", metrics_from_counts(*confusion_counts(zeros, zeros)),
      {"precision": 1, "recall": 1, "f1": 1, "iou": 1, "accuracy": 1})
check("all change, nothing predicted", metrics_from_counts(*confusion_counts(zeros, ones)),
      {"precision": 0, "recall": 0, "f1": 0, "iou": 0, "accuracy": 0})
check("no change, everything predicted", metrics_from_counts(*confusion_counts(ones, zeros)),
      {"precision": 0, "recall": 0, "f1": 0, "iou": 0, "accuracy": 0})
check("perfect prediction", metrics_from_counts(*confusion_counts(target, target)),
      {"precision": 1, "recall": 1, "f1": 1, "iou": 1, "accuracy": 1})

# 4. Accumulating random batches equals computing on everything at once
rng = np.random.default_rng(0)
big_pred = rng.integers(0, 2, size=(10, 32, 32))
big_target = rng.integers(0, 2, size=(10, 32, 32))
acc = ConfusionAccumulator()
for start in range(0, 10, 3):  # uneven batches: 3, 3, 3, 1
    acc.update(big_pred[start:start + 3], big_target[start:start + 3])
check("batched == all at once", acc.compute(), metrics_from_counts(*confusion_counts(big_pred, big_target)))

# 5. Probabilities are rejected instead of silently miscounted
try:
    confusion_counts(np.array([[0.2, 0.9]]), np.array([[0, 1]]))
    raise AssertionError("non-binary input was accepted")
except ValueError:
    print("OK   non-binary input rejected")

# Per-image metrics for a small batch
print()
for row in per_image_metrics(np.stack([pred, zeros]), np.stack([target, zeros]), names=["toy", "empty"]):
    print(f"{row['name']:>6}: {format_metrics(row)}")

## Task 3 · Data preparation and dataset

**Source.** LEVIR-CD from the Kaggle mirror [`mdrifaturrahman33/levir-cd`](https://www.kaggle.com/datasets/mdrifaturrahman33/levir-cd), laid out as `LEVIR CD/{train,val,test}/{A,B,label}/*.png` with the official 445/64/128 split. If a copy already sits in `PATHS.raw` on Drive, that one is used instead.

**Tiling (one time, about 10 minutes).** Each 1024×1024 A/B/label triple becomes 16 non-overlapping 256×256 tiles named `{image}_{row}_{col}.png`, giving 7,120 / 1,024 / 2,048 tiles. Labels are stored as 0/1. The tiles go to `tiles.zip` on Drive (about 2.5 GB).

**Every session after that.** `tiles.zip` is copied from Drive to the VM's local disk and unzipped (about 1–2 minutes), because training reads thousands of small files and Drive is slow at that. The raw download isn't needed again.

**Kaggle access.** Public datasets usually download without signing in. If the download fails, add two Colab secrets (key icon on the left), `KAGGLE_USERNAME` and `KAGGLE_KEY`, from kaggle.com → Settings → API → Create New Token, then rerun.

In [ ]:
import json
import re
import shutil
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor

from PIL import Image

CONFIG["data"] = {
    "kaggle_dataset": "mdrifaturrahman33/levir-cd",
    "splits": {"train": 445, "val": 64, "test": 128},  # official image counts
    "image_size": 1024,
    "tile_size": 256,
    "label_dirs": ("label", "OUT", "labels"),  # names used by different LEVIR-CD copies
    "io_workers": 8,  # threads for reading/writing PNGs while tiling
}
DATA = CONFIG["data"]
TILES_PER_IMAGE = (DATA["image_size"] // DATA["tile_size"]) ** 2


def natural_key(s):
    """Sort 'train_2' before 'train_10'."""
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", str(s))]


def find_levir_root(base):
    """Return the folder that holds train/val/test, searching below base."""
    base = Path(base)
    if (base / "train" / "A").is_dir():
        return base
    for a_dir in sorted(base.rglob("A")):
        if a_dir.is_dir() and a_dir.parent.name == "train" and (a_dir.parent.parent / "val").is_dir():
            return a_dir.parent.parent
    return None


def label_dir(split_dir):
    for name in DATA["label_dirs"]:
        if (split_dir / name).is_dir():
            return split_dir / name
    raise FileNotFoundError(f"No label folder ({', '.join(DATA['label_dirs'])}) in {split_dir}")


def _kaggle_credentials_from_secrets():
    """Copy KAGGLE_USERNAME / KAGGLE_KEY from Colab secrets into the environment, if set."""
    try:
        from google.colab import userdata
    except ImportError:
        return
    for key in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        if key not in os.environ:
            try:
                os.environ[key] = userdata.get(key)
            except Exception:
                pass  # secret not set; public download may still work


def locate_raw_dataset():
    """Use LEVIR-CD from Drive if present, otherwise download it from Kaggle to local disk."""
    root = find_levir_root(PATHS.raw)
    if root is not None:
        print("Using LEVIR-CD from Drive:", root)
        return root
    try:
        import kagglehub
    except ImportError as e:
        raise RuntimeError("kagglehub is missing. Run `!pip install kagglehub` and rerun this cell.") from e
    _kaggle_credentials_from_secrets()
    print(f"Downloading {DATA['kaggle_dataset']} from Kaggle (about 2.5 GB, one time)...")
    try:
        path = kagglehub.dataset_download(DATA["kaggle_dataset"])
    except Exception as e:
        raise RuntimeError(
            "Kaggle download failed. Add Colab secrets KAGGLE_USERNAME and KAGGLE_KEY "
            "(kaggle.com > Settings > API > Create New Token) and rerun."
        ) from e
    root = find_levir_root(path)
    if root is None:
        raise FileNotFoundError(f"Downloaded to {path}, but found no train/A folder inside.")
    print("Downloaded to:", root)
    return root


def list_split(root, split):
    """Sorted image names for a split, after checking A, B and label hold the same files."""
    split_dir = Path(root) / split
    a = {p.name for p in (split_dir / "A").glob("*.png")}
    b = {p.name for p in (split_dir / "B").glob("*.png")}
    lab = {p.name for p in label_dir(split_dir).glob("*.png")}
    if not (a == b == lab):
        raise ValueError(f"{split}: A/B/label file names differ (A={len(a)}, B={len(b)}, label={len(lab)})")
    expected = DATA["splits"][split]
    if len(a) != expected:
        print(f"WARNING: {split} has {len(a)} images, expected {expected}")
    return sorted(a, key=natural_key)

In [ ]:
def load_rgb(path):
    return np.asarray(Image.open(path).convert("RGB"))


def load_label(path):
    """Binary uint8 mask (0/1). Handles labels stored as 0/255 or already 0/1."""
    m = np.asarray(Image.open(path).convert("L"))
    return (m > (127 if m.max() > 1 else 0)).astype(np.uint8)


def iter_tiles(arr, tile):
    """Yield (row, col, tile) for non-overlapping tile x tile crops in row-major order."""
    h, w = arr.shape[:2]
    if h % tile or w % tile:
        raise ValueError(f"Image {h}x{w} is not divisible into {tile}x{tile} tiles")
    for r in range(h // tile):
        for c in range(w // tile):
            yield r, c, arr[r * tile:(r + 1) * tile, c * tile:(c + 1) * tile]


def _tile_one(job):
    root, split, name, out_dir = job
    split_dir = root / split
    a = load_rgb(split_dir / "A" / name)
    b = load_rgb(split_dir / "B" / name)
    m = load_label(label_dir(split_dir) / name)
    size = DATA["image_size"]
    for arr, kind in ((a, "A"), (b, "B"), (m, "label")):
        if arr.shape[:2] != (size, size):
            raise ValueError(f"{split}/{kind}/{name} is {arr.shape[:2]}, expected {size}x{size}")
    stem = Path(name).stem
    tile = DATA["tile_size"]
    for (r, c, ta), (_, _, tb), (_, _, tm) in zip(iter_tiles(a, tile), iter_tiles(b, tile), iter_tiles(m, tile)):
        tile_name = f"{stem}_{r}_{c}.png"
        Image.fromarray(ta).save(out_dir / split / "A" / tile_name)
        Image.fromarray(tb).save(out_dir / split / "B" / tile_name)
        Image.fromarray(tm).save(out_dir / split / "label" / tile_name)


def build_tiles_zip(root):
    """Tile every split on local disk, zip the tiles, then copy the zip to Drive."""
    work = PATHS.local / "tiles_build"
    if work.exists():
        shutil.rmtree(work)
    jobs = []
    for split in DATA["splits"]:
        for kind in ("A", "B", "label"):
            (work / split / kind).mkdir(parents=True)
        jobs += [(Path(root), split, name, work) for name in list_split(root, split)]

    t0 = time.time()
    with ThreadPoolExecutor(DATA["io_workers"]) as pool:
        for i, _ in enumerate(pool.map(_tile_one, jobs), 1):
            if i % 100 == 0 or i == len(jobs):
                print(f"  tiled {i}/{len(jobs)} images ({time.time() - t0:.0f}s)")

    # PNGs are already compressed, so store them in the zip without recompressing.
    local_zip = PATHS.local / "tiles.zip"
    with zipfile.ZipFile(local_zip, "w", zipfile.ZIP_STORED) as zf:
        for p in sorted(work.rglob("*.png")):
            zf.write(p, p.relative_to(work))
    print(f"  zipped {local_zip.stat().st_size / 1e9:.2f} GB ({time.time() - t0:.0f}s)")

    # Copy under a temporary name, then rename, so a disconnect never leaves a half-written zip.
    PATHS.tiles_zip.parent.mkdir(parents=True, exist_ok=True)
    partial = PATHS.tiles_zip.with_name(PATHS.tiles_zip.name + ".partial")
    shutil.copyfile(local_zip, partial)
    os.replace(partial, PATHS.tiles_zip)
    local_zip.unlink()
    print(f"  saved {PATHS.tiles_zip} ({time.time() - t0:.0f}s)")

    # The freshly built tiles become this session's local copy.
    if PATHS.local_tiles.exists():
        shutil.rmtree(PATHS.local_tiles)
    work.rename(PATHS.local_tiles)
    (PATHS.local_tiles / ".complete").touch()


def ensure_local_tiles():
    """Make sure tiles are on local disk: reuse, unzip from Drive, or build from raw data."""
    marker = PATHS.local_tiles / ".complete"
    if marker.exists():
        print("Tiles already on local disk:", PATHS.local_tiles)
        return
    if not PATHS.tiles_zip.exists():
        print("No tiles.zip on Drive yet. Building it (one time).")
        build_tiles_zip(locate_raw_dataset())
        return
    t0 = time.time()
    print("Copying tiles.zip from Drive to local disk...")
    local_zip = PATHS.local / "tiles.zip"
    shutil.copyfile(PATHS.tiles_zip, local_zip)
    if PATHS.local_tiles.exists():
        shutil.rmtree(PATHS.local_tiles)
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(PATHS.local_tiles)
    local_zip.unlink()
    marker.touch()
    print(f"  unzipped to {PATHS.local_tiles} ({time.time() - t0:.0f}s)")

In [ ]:
ensure_local_tiles()

print()
for split, n_images in DATA["splits"].items():
    counts = {kind: len(list((PATHS.local_tiles / split / kind).glob("*.png"))) for kind in ("A", "B", "label")}
    expected = n_images * TILES_PER_IMAGE
    status = "OK  " if all(v == expected for v in counts.values()) else "FAIL"
    print(f"{status} {split:<5} A={counts['A']}  B={counts['B']}  label={counts['label']}  (expected {expected})")

In [ ]:
from torch.utils.data import DataLoader, Dataset


class PairedAugment:
    """Random horizontal/vertical flips and 90° rotations, identical for A, B and the mask."""

    def __init__(self, hflip=True, vflip=True, rot90=True):
        self.hflip, self.vflip, self.rot90 = hflip, vflip, rot90

    def __call__(self, a, b, m):
        if self.hflip and torch.rand(()) < 0.5:
            a, b, m = (x.flip(-1) for x in (a, b, m))
        if self.vflip and torch.rand(()) < 0.5:
            a, b, m = (x.flip(-2) for x in (a, b, m))
        if self.rot90:
            k = int(torch.randint(0, 4, ()))
            if k:
                a, b, m = (torch.rot90(x, k, (-2, -1)) for x in (a, b, m))
        return a, b, m


class Normalize:
    """Per-channel (x - mean) / std for a (3, H, W) image in [0, 1]."""

    def __init__(self, mean, std):
        self.mean = torch.tensor(mean, dtype=torch.float32).view(-1, 1, 1)
        self.std = torch.tensor(std, dtype=torch.float32).view(-1, 1, 1)

    def __call__(self, x):
        return (x - self.mean) / self.std


def load_stats():
    """Training-set channel mean/std from Task 4, or None if not computed yet."""
    if PATHS.stats_file.exists():
        return json.loads(PATHS.stats_file.read_text())
    return None


class ChangeDetectionDataset(Dataset):
    """256x256 LEVIR-CD tiles.

    Returns (A, B, mask, name): A and B are float32 (3, H, W), in [0, 1] or normalized
    if stats are given. mask is float32 (1, H, W) with values 0/1.
    """

    def __init__(self, split, root=None, transform=None, stats=None):
        self.dir = Path(root or PATHS.local_tiles) / split
        self.names = sorted((p.name for p in (self.dir / "A").glob("*.png")), key=natural_key)
        if not self.names:
            raise FileNotFoundError(f"No tiles in {self.dir / 'A'}. Run ensure_local_tiles() first.")
        self.transform = transform
        self.normalize = Normalize(stats["mean"], stats["std"]) if stats else None

    def __len__(self):
        return len(self.names)

    @staticmethod
    def _image(path):
        arr = np.asarray(Image.open(path).convert("RGB"), dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1)

    def __getitem__(self, i):
        name = self.names[i]
        a = self._image(self.dir / "A" / name)
        b = self._image(self.dir / "B" / name)
        m = torch.from_numpy(np.asarray(Image.open(self.dir / "label" / name), dtype=np.float32))[None]
        if self.transform is not None:
            a, b, m = self.transform(a, b, m)
        if self.normalize is not None:
            a, b = self.normalize(a), self.normalize(b)
        return a, b, m, name


def make_loader(split, batch_size, train=False, stats=None, num_workers=2):
    """DataLoader with augmentation, shuffling and drop_last for training only."""
    ds = ChangeDetectionDataset(split, transform=PairedAugment() if train else None, stats=stats)
    gen = torch.Generator()
    gen.manual_seed(CONFIG["seed"])
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=train,
        drop_last=train,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker,
        generator=gen,
        persistent_workers=num_workers > 0,
    )

### Demo: a training batch

Loads one augmented training batch, checks shapes, types and mask values, checks that A, B and the mask get the same flips and rotations, and shows A, B and the mask side by side. The figure is saved to `results/figures/task3_sample_batch.png`.

In [ ]:
import matplotlib.pyplot as plt

for split in DATA["splits"]:
    print(f"{split:<5} {len(ChangeDetectionDataset(split)):>5} tiles")

# Unnormalized so the images display with natural colours.
loader = make_loader("train", batch_size=4, train=True, num_workers=2)
a, b, m, names = next(iter(loader))
print(f"\nA {tuple(a.shape)} {a.dtype} range [{a.min():.2f}, {a.max():.2f}]")
print(f"B {tuple(b.shape)} {b.dtype}")
print(f"mask {tuple(m.shape)} {m.dtype} values {sorted(m.unique().tolist())}")
assert a.shape == b.shape == (4, 3, 256, 256) and m.shape == (4, 1, 256, 256)
assert set(m.unique().tolist()) <= {0.0, 1.0}

# Paired augmentation: feed identical, asymmetric content as A, B and mask, check it stays aligned.
probe = torch.arange(16.0).view(1, 4, 4)
aug = PairedAugment()
for _ in range(20):
    pa, pb, pm = aug(probe.repeat(3, 1, 1), probe.repeat(3, 1, 1), probe.clone())
    assert torch.equal(pa, pb) and torch.equal(pa[0], pm[0])
print("OK   A, B and mask receive identical flips/rotations")

fig, axes = plt.subplots(len(names), 3, figsize=(9, 3 * len(names)))
for i, name in enumerate(names):
    for ax, img, title in zip(axes[i], (a[i], b[i], m[i]), ("A (before)", "B (after)", "change mask")):
        if img.shape[0] == 3:
            ax.imshow(img.permute(1, 2, 0).numpy())
        else:
            ax.imshow(img[0].numpy(), cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"{title}\n{name}" if title.startswith("A") else title, fontsize=9)
        ax.axis("off")
    axes[i, 2].text(5, 20, f"{m[i].mean().item():.1%} changed", color="yellow", fontsize=9)
plt.tight_layout()
fig.savefig(PATHS.figures / "task3_sample_batch.png", dpi=120, bbox_inches="tight")
plt.show()

## Task 4 · Exploratory data analysis

Reads the 256×256 tiles from Task 3 (local disk) and reports:
- image and tile counts per split
- the fraction of changed pixels per split and per image, and how imbalanced the classes are
- the share of tiles with no change at all
- example tiles with large, small and no change

It then computes the per-channel mean and std of the training images (A and B together, so both dates share one normalization) and saves them to `stats.json` on Drive. The dataset uses this file from here on. The stats are computed once and reused in later sessions; set `CONFIG["eda"]["recompute_stats"] = True` to redo them.

Figures go to `results/figures/` and a summary to `results/eda_summary.json`.

In [ ]:
CONFIG["eda"] = {
    "io_workers": 8,
    "recompute_stats": False,  # True recomputes stats.json even if it exists
}


def image_of_tile(tile_name):
    """'train_12_3_1.png' -> 'train_12' (the original 1024x1024 image)."""
    return Path(tile_name).stem.rsplit("_", 2)[0]


def _changed_pixels(path):
    m = np.asarray(Image.open(path))
    return int(m.sum()), int(m.size)


def label_statistics(split):
    """Changed-pixel counts for every tile in a split, plus per-image totals."""
    paths = sorted((PATHS.local_tiles / split / "label").glob("*.png"), key=natural_key)
    with ThreadPoolExecutor(CONFIG["eda"]["io_workers"]) as pool:
        counts = list(pool.map(_changed_pixels, paths))
    tile_names = [p.name for p in paths]
    tile_changed = np.array([c for c, _ in counts], dtype=np.int64)
    tile_pixels = np.array([n for _, n in counts], dtype=np.int64)

    per_image = {}
    for name, c, n in zip(tile_names, tile_changed, tile_pixels):
        img = image_of_tile(name)
        changed, pixels = per_image.get(img, (0, 0))
        per_image[img] = (changed + int(c), pixels + int(n))
    image_names = sorted(per_image, key=natural_key)
    return {
        "tile_names": tile_names,
        "tile_frac": tile_changed / tile_pixels,
        "image_names": image_names,
        "image_frac": np.array([per_image[k][0] / per_image[k][1] for k in image_names]),
        "changed": int(tile_changed.sum()),
        "pixels": int(tile_pixels.sum()),
    }


def _channel_sums(path):
    x = np.asarray(Image.open(path).convert("RGB"), dtype=np.float64).reshape(-1, 3) / 255.0
    return x.sum(0), (x * x).sum(0), x.shape[0]


def channel_sums(split, kind):
    """Per-channel sum, sum of squares and pixel count over one image folder."""
    paths = sorted((PATHS.local_tiles / split / kind).glob("*.png"))
    s, sq, n = np.zeros(3), np.zeros(3), 0
    with ThreadPoolExecutor(CONFIG["eda"]["io_workers"]) as pool:
        for a, b, c in pool.map(_channel_sums, paths):
            s += a
            sq += b
            n += c
    return s, sq, n


def mean_std(s, sq, n):
    mean = s / n
    std = np.sqrt(np.maximum(sq / n - mean**2, 0.0))
    return mean, std


def compute_train_stats():
    """Channel mean/std over train A and B tiles, combined and per date."""
    sums = {kind: channel_sums("train", kind) for kind in ("A", "B")}
    combined = tuple(sums["A"][i] + sums["B"][i] for i in range(3))
    mean, std = mean_std(*combined)
    stats = {
        "mean": mean.round(6).tolist(),
        "std": std.round(6).tolist(),
        "per_date": {},
        "source": "train tiles, A and B combined, RGB in [0, 1]",
        "pixels": int(combined[2]),
    }
    for kind, s in sums.items():
        m, sd = mean_std(*s)
        stats["per_date"][kind] = {"mean": m.round(6).tolist(), "std": sd.round(6).tolist()}
    return stats


def write_json(obj, path):
    """Write JSON via a temporary file, so a disconnect never leaves a half-written file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".partial")
    tmp.write_text(json.dumps(obj, indent=2))
    os.replace(tmp, path)

In [ ]:
t0 = time.time()
LABEL_STATS = {split: label_statistics(split) for split in DATA["splits"]}
print(f"Read label tiles in {time.time() - t0:.0f}s")
print()

header = f"{'split':<6}{'images':>7}{'tiles':>7}{'changed px':>12}{'no-change tiles':>17}{'median img':>12}{'max img':>9}"
print(header)
print("-" * len(header))
eda_summary = {}
for split, st in LABEL_STATS.items():
    row = {
        "images": len(st["image_names"]),
        "tiles": len(st["tile_names"]),
        "changed_pixel_fraction": st["changed"] / st["pixels"],
        "no_change_tile_fraction": float((st["tile_frac"] == 0).mean()),
        "no_change_image_fraction": float((st["image_frac"] == 0).mean()),
        "median_image_changed_fraction": float(np.median(st["image_frac"])),
        "max_image_changed_fraction": float(st["image_frac"].max()),
    }
    eda_summary[split] = row
    print(f"{split:<6}{row['images']:>7}{row['tiles']:>7}{row['changed_pixel_fraction']:>12.2%}"
          f"{row['no_change_tile_fraction']:>17.1%}{row['median_image_changed_fraction']:>12.2%}"
          f"{row['max_image_changed_fraction']:>9.1%}")

train = LABEL_STATS["train"]
neg_pos_ratio = (train["pixels"] - train["changed"]) / max(train["changed"], 1)
eda_summary["train_negative_to_positive_ratio"] = neg_pos_ratio
print()
print(f"Train class ratio: {neg_pos_ratio:.1f} unchanged pixels per changed pixel")
print("(Task 6 loss: BCE + Dice. Dice is not swamped by this imbalance; a BCE pos_weight could use this ratio.)")

write_json(eda_summary, PATHS.results / "eda_summary.json")
print("Saved", PATHS.results / "eda_summary.json")

In [ ]:
import matplotlib.pyplot as plt

# 1. Per-image changed fraction, one panel per split
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=False)
for ax, (split, st) in zip(axes, LABEL_STATS.items()):
    ax.hist(st["image_frac"] * 100, bins=30, color="tab:blue", edgecolor="white")
    ax.axvline(np.median(st["image_frac"]) * 100, color="tab:red", ls="--", label="median")
    ax.set_title(f"{split}: {len(st['image_names'])} images, {st['changed'] / st['pixels']:.2%} changed")
    ax.set_xlabel("changed pixels per image (%)")
    ax.legend()
axes[0].set_ylabel("images")
plt.tight_layout()
fig.savefig(PATHS.figures / "task4_image_change_hist.png", dpi=120, bbox_inches="tight")
plt.show()

# 2. Per-tile changed fraction in train (log scale: most tiles have little or no change)
tile_frac = train["tile_frac"]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(tile_frac[tile_frac > 0] * 100, bins=50, color="tab:green", edgecolor="white")
ax.set_yscale("log")
ax.set_xlabel("changed pixels per tile (%)")
ax.set_ylabel("tiles (log scale)")
ax.set_title(f"train tiles with change ({(tile_frac == 0).mean():.1%} of tiles have none and are not shown)")
plt.tight_layout()
fig.savefig(PATHS.figures / "task4_tile_change_hist.png", dpi=120, bbox_inches="tight")
plt.show()

# 3. Example train tiles: most change, a typical changed tile, and no change
order = np.argsort(tile_frac)
changed_idx = order[tile_frac[order] > 0]
examples = {
    "most change": order[-1],
    "typical changed tile": changed_idx[len(changed_idx) // 2],
    "no change": order[0],
}
fig, axes = plt.subplots(len(examples), 3, figsize=(9, 3 * len(examples)))
train_dir = PATHS.local_tiles / "train"
for row, (label, idx) in zip(axes, examples.items()):
    name = train["tile_names"][idx]
    images = (Image.open(train_dir / "A" / name), Image.open(train_dir / "B" / name), Image.open(train_dir / "label" / name))
    titles = (f"A · {label}\n{name}", "B", f"mask · {tile_frac[idx]:.1%} changed")
    for ax, img, title in zip(row, images, titles):
        if img.mode == "L":
            ax.imshow(np.asarray(img), cmap="gray", vmin=0, vmax=1)
        else:
            ax.imshow(img)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
plt.tight_layout()
fig.savefig(PATHS.figures / "task4_example_tiles.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
if PATHS.stats_file.exists() and not CONFIG["eda"]["recompute_stats"]:
    print("Using existing", PATHS.stats_file)
else:
    t0 = time.time()
    print("Computing channel mean/std over train A and B tiles...")
    write_json(compute_train_stats(), PATHS.stats_file)
    print(f"Saved {PATHS.stats_file} ({time.time() - t0:.0f}s)")

STATS = load_stats()
print()
print(f"{'':<10}{'R':>8}{'G':>8}{'B':>8}")
print(f"{'mean':<10}" + "".join(f"{v:>8.4f}" for v in STATS["mean"]))
print(f"{'std':<10}" + "".join(f"{v:>8.4f}" for v in STATS["std"]))
for kind in ("A", "B"):
    print(f"{'mean ' + kind:<10}" + "".join(f"{v:>8.4f}" for v in STATS["per_date"][kind]["mean"]))
print("(mean A vs mean B shows the overall brightness/colour shift between the two dates)")

### Demo: normalized batches

Loads a few unaugmented training batches with the new stats. After normalization each channel should have a mean near 0 and a std near 1. They won't be exact because only a sample of batches is checked.

In [ ]:
check_loader = make_loader("train", batch_size=32, train=False, stats=STATS, num_workers=2)
s, sq, n = torch.zeros(3), torch.zeros(3), 0
for i, (a, b, m, _) in enumerate(check_loader):
    x = torch.cat([a, b]).double()
    s += x.sum(dim=(0, 2, 3)).float()
    sq += (x * x).sum(dim=(0, 2, 3)).float()
    n += x.shape[0] * x.shape[2] * x.shape[3]
    if i == 9:  # 10 batches = 640 images
        break
mean = s / n
std = (sq / n - mean**2).clamp_min(0).sqrt()
print("normalized mean:", [round(v, 3) for v in mean.tolist()])
print("normalized std: ", [round(v, 3) for v in std.tolist()])
assert torch.isfinite(mean).all() and torch.isfinite(std).all()
assert (mean.abs() < 0.5).all() and ((std - 1).abs() < 0.5).all(), "normalization looks wrong"
print("OK   dataset loads stats.json and normalizes A and B")